In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib agg
    
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
       
from HPIB.HP4155 import HP4155
from HPIB.HPT import Plot, PlotVgs, PlotVp, CalcIsSat, SecDer, Plot2P
from HPIB.DevParams import UMC
from HPIB.INOSerial import Arduino

from BFmodule import DR

from YFunc import YFuncExtraction

from Test import TestDevice, WriteLog

from IPython.display import clear_output, display
from os import makedirs, rename
from time import sleep
from datetime import datetime, timedelta

Elsa=DR()

HP=HP4155("GPIB0::17", debug=False)
HP.IntTime="LONG"

INO=Arduino("COM3")
print(INO.ask('*'.encode()))

Elsa v2.4.2
HEWLETT-PACKARD,4155A,0,01.04:01.04:01.00
InoMatrix



In [2]:
def WriteLog(msg, path, mode='a', end='\n', output=True):
    with open(path, mode) as logfile:
        logfile.write(msg+end)
    if output:
        print(f"{msg}{end}", end='')

def TestDevice(device, chn, path, params, HiPot=False):
    global HP, INO, Elsa, prog_bar
    INO.opench(chn+1)

    if not device:
        INO.opench(0)
        sleep(2)
        return 0

    if device[:2].upper() not in ['CA', 'CB', 'CG', 'TP', 'TN', 'DP', 'DN']:
        return "Invalid device"

    WriteLog(f"## Ch {chn+1} {device}", path + 'log.txt')
    pathp=path+device
    makedirs(pathp, exist_ok=True)
    
    ####################### Measure Diode
    #
    #
    if 'D' in device.upper():
        
        HP.StopCond="COMP"
        HP.IntTime="MED"
        HP.SingleDiode(params['Vfmin'], params['Vfmax'], params['Vfstep'], 'SMU2', 'SMU1', Comp=params['IComp'] if not HiPot else 2*params['IComp'])
            
        now=datetime.now().strftime('%y%m%d %H%M%S')
        Plot(HP.SingleSave(f"{pathp}/{now}.csv", timeout=30, real=True), 'Vf', 'If')
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            rename(f"{pathp}/{now}.csv", f"{pathp}/Diode - {temp} - {now}.csv")
            rename(f"{pathp}/{now}.png", f"{pathp}/Diode - {temp} - {now}.png")
        except Exception as e:
            print (">>> Error:", e)
        
        now=datetime.now()

        # with open(f"{pathp}/V100uA.log", 'a') as DiodeParam:
        #     DiodeParam.write(f"{temp},{format(RCB, '.2e')}\n")
            
        while (datetime.now()-now).seconds < params['min_wait']:
            prog_bar.update(f"Waiting: {params['min_wait']-(datetime.now()-start).seconds} s")
            sleep(0.5)
        prog_bar.update("Measuring")

        HP.SingleDiode(params['Vfmin'], params['Vfmax'], params['Vfstep'], 'SMU4', 'SMU3', Comp=params['IComp'] if not HiPot else 2*params['IComp'])
            
        now=datetime.now().strftime('%y%m%d %H%M%S')
        Plot(HP.SingleSave(f"{pathp}/{now}.csv", timeout=30, real=True), 'Vf', 'If')
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            rename(f"{pathp}/{now}.csv", f"{pathp}/1N4007 - {temp} - {now}.csv")
            rename(f"{pathp}/{now}.png", f"{pathp}/1N4007 - {temp} - {now}.png")
        except Exception as e:
            print (">>> Error:", e)
        
        now=datetime.now()

        # with open(f"{pathp}/V100uA.log", 'a') as DiodeParam:
        #     DiodeParam.write(f"{temp},{format(RCB, '.2e')}\n")
            
        while (datetime.now()-now).seconds < params['min_wait']:
            prog_bar.update(f"Waiting: {params['min_wait']-(datetime.now()-start).seconds} s")
            sleep(0.5)
        prog_bar.update("Measuring")
            
        HP.StopCond="OFF"
        HP.IntTime="LONG"
        
        INO.opench(0)
        WriteLog('', path + 'log.txt')
        return 0
    #
    #
    #######################

    ####################### Measure Transistor
    #
    #

    if 'T' in device.upper():
        
        ptype='P' in device.upper()
        
        HP.SetVgs(params['Vmin'], params['Vmax'], params['Vstep'], params['Vd'], ptype=ptype)
  
        now=datetime.now().strftime('%y%m%d %H%M')
        LIN = PlotVgs(HP.SingleSave(f"{pathp}/IdVgs - {now}.csv", timeout=30))
        
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            newname=f"{pathp}/IdVgs - {temp} - {now}"
            rename(f"{pathp}/IdVgs - {now}.csv", f"{newname}.csv")
            rename(f"{pathp}/IdVgs - {now}.png", f"{newname}.png")
        except Exception as e:
            print (">>> Error:", e)
        now=datetime.now()

        try:
            LIN, Vth, SS, migm, miyf, theta1, theta2, errmax = YFuncExtraction(f"{newname}.csv", UMC[int(device[2:])], 4.2, 3.9, params['Vd'])
            WriteLog(f"LIN={format(LIN, '.3f')}, Vth={format(Vth, '.3f')} V, SS={format(SS, '.2f')} mV/dec, miyf={format(migm, '.1f')}, miyf={format(miyf, '.1f')}, theta1={format(theta1, '.3e')}, theta2={format(theta2, '.3e')}", path + 'log.txt')
            WriteLog(f"{temp},{format(LIN, '.3f')},{format(Vth, '.3f')},{format(SS, '.2f')},{format(migm, '.1f')},{format(miyf, '.1f')},{format(theta1, '.3e')},{format(theta2, '.3e')}", pathp + '/params.txt')
        except Exception as e:
            print (">>> Error:", e)
            WriteLog(f"Vth={LIN} V", path + 'log.txt')
        
        while (datetime.now()-now).seconds < params['min_wait']:
            prog_bar.update(f"Waiting: {params['min_wait']-(datetime.now()-now).seconds} s")
            sleep(0.5)
        prog_bar.update("Measuring")
            
        if HiPot:
            now=datetime.now().strftime('%y%m%d %H%M')
            HP.SetVgs(params['Vmin'], params['Vmax'], params['Vstep'], params['Vmax'], ptype=ptype, sat=True)
            HP.SingleSave(f"{pathp}/IdVgsSat - {now}.csv", timeout=30)
            try:
                temp=format(Elsa.GetT('t4k'), '07.3f')
                rename(f"{pathp}/IdVgsSat - {now}.csv", f"{pathp}/IdVgsSat - {temp} - {now}.csv")
            except Exception as e:
                print(">>> Error:", e)
            n, Ispec = CalcIsSat(f"{pathp}/IdVgsSat - {temp} - {now}.csv",temp)
            WriteLog(f"n={format(n, '.3f')}, Ispec={format(Ispec, '.3e')} A", path + 'log.txt')

            now=datetime.now()
            while (datetime.now()-now).seconds < 4*params['min_wait']:
                prog_bar.update(f"Waiting: {4*params['min_wait']-(datetime.now()-start).seconds} s")
                sleep(0.5)
            prog_bar.update("Measuring")
            
            if Ispec != 0:
                now=datetime.now().strftime('%y%m%d %H%M')
                HP.SetVp(Ispec, params['Vmin'], params['Vmax'], 0.05, ptype=ptype)
                HP.SingleSave(f"{pathp}/VpVg - {now}.csv", timeout=30)
                try:
                    temp=format(Elsa.GetT('t4k'), '07.3f')
                    rename(f"{pathp}/VpVg - {now}.csv", f"{pathp}/VpVg - {temp} - {now}.csv")
                except Exception as e:
                    print (">>> Error:", e)
                VTO=PlotVp(f"{pathp}/VpVg - {temp} - {now}.csv")
                WriteLog(f"VTO={VTO} V", path + 'log.txt')

                now=datetime.now()
                while (datetime.now()-now).seconds < 4*params['min_wait']:
                    prog_bar.update(f"Waiting: {4*params['min_wait']-(datetime.now()-now).seconds} s")
                    sleep(0.5)
                prog_bar.update("Measuring")

            now=datetime.now().strftime('%y%m%d %H%M')
            HP.SetVds(params['Vmin'], params['Vmax'], params['Vstep'], params['Vgmin'], params['Vgmax'], params['Vgstep'], ptype=ptype)
            HP.SingleSave(f"{pathp}/IdVds - {now}.csv", timeout=30)
            try:
                temp=format(Elsa.GetT('t4k'), '07.3f')
                rename(f"{pathp}/IdVds - {now}.csv", f"{pathp}/IdVds - {temp} - {now}.csv")
            except Exception as e:
                print (">>> Error:", e)
            Plot(f"{pathp}/IdVds - {temp} - {now}.csv", 'Vd', 'Id')
            
            now=datetime.now()
            while (datetime.now()-now).seconds < 4*params['min_wait']:
                prog_bar.update(f"Waiting: {4*params['min_wait']-(datetime.now()-now).seconds} s")
                sleep(0.5)
            prog_bar.update("Measuring")
            
        INO.opench(0)
        WriteLog('', path + 'log.txt')
        return 0
    #
    #
    #######################
        
    ####################### Measure CrossBridge
    #
    #
    if 'CG' in device.upper():
           
        now=datetime.now().strftime('%y%m%d %H%M')
        V10u=HP.MeasV10u(f"{pathp}/2P - {now}.csv", timeout=0.5)
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            rename(f"{pathp}/2P - {now}.csv", f"{pathp}/2P - {temp} - {now}.csv")
            rename(f"{pathp}/2P - {now}.png", f"{pathp}/2P - {temp} - {now}.png")
        except Exception as e:
            print (">>> Error:", e)
        WriteLog(f"V_10u={format(V10u, '.2f')}", path + 'log.txt')
        with open(f"{pathp}/VxT 10uA.log", 'a') as VxT:
            VxT.write(f"{temp},{format(V10u, '.4e')}\n")

    if 'CA' in device.upper():

        HP.Set2P(params['Ifmin']/params['Iffactor'], params['Ifmax']/params['Iffactor'], params['Ifpoints'], SMUN='SMU1', SMUP='SMU2', Comp=params['VComp'])
            
        now=datetime.now().strftime('%y%m%d %H%M')
        Rshort=Plot2P(HP.SingleSave(f"{pathp}/2P - {now}.csv", timeout=30))
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            rename(f"{pathp}/2P - {now}.csv", f"{pathp}/2P - {temp} - {now}.csv")
            rename(f"{pathp}/2P - {now}.png", f"{pathp}/2P - {temp} - {now}.png")
        except Exception as e:
            print (">>> Error:", e)
        WriteLog(f"Rshort={format(Rshort, '.2f')}", path + 'log.txt')
        with open(f"{pathp}/RxT 2P.log", 'a') as RxT2p:
            RxT2p.write(f"{temp},{format(Rshort, '.2f')}\n")

    if 'CB' in device.upper():
    
        HP.Set4P(params['Ifmin'], params['Ifmax'], params['Ifpoints'], Im='SMU1', Ip='SMU2', Comp=params['VComp'])
        
        now=datetime.now().strftime('%y%m%d %H%M')
        RCB=Plot2P(HP.SingleSave(f"{pathp}/4P - {now}.csv", timeout=30))
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            rename(f"{pathp}/4P - {now}.csv", f"{pathp}/4P - {temp} - {now}.csv")
            rename(f"{pathp}/4P - {now}.png", f"{pathp}/4P - {temp} - {now}.png")
        except Exception as e:
            print (">>> Error:", e)
        WriteLog(f"RCB={format(RCB, '.2e')}", path + 'log.txt')
        with open(f"{pathp}/RxT 4P.log", 'a') as RxT4p:
            RxT4p.write(f"{temp},{format(RCB, '.2e')}\n")

    now=datetime.now()
    while (datetime.now()-now).seconds < params['min_wait']:
        prog_bar.update(f"Waiting: {params['min_wait']-(datetime.now()-now).seconds} s")
        sleep(0.5)
    prog_bar.update("Measuring")
            
    INO.opench(0)
    WriteLog('', path + 'log.txt')
    return 0
    #
    #
    #######################

In [3]:
params = {
'Vd' : 0.05,
'Vmin' : 0,
'Vmax' : 1.5,
'Vstep' : 0.02,

'Vgmin' : 0.6,
'Vgmax' : 1.4,
'Vgstep' : 0.2,

'Vfmin' : 0,
'Vfmax' : -2,
'Vfstep' : -0.01,
'IComp' : 0.5e-3,

'Ifmin' : -5e-3,
'Ifmax' : 5e-3,
'Ifpoints' : 50,
'Iffactor' : 50,
'VComp' : 1.5,

'min_wait' : 15
}

prepath="C:/Users/Zucchi/Documents/Medidas/241129 TP7 TP9 TP5 TN3/"
DeviceList = ['TP7', 'TP9', 'TP5', 'TN3']
# DeviceList = ['', '', '', 'TN4', '', '']

In [ ]:
###### Pre measurement

if Elsa.GetT('t4k') > 150:
    path = prepath+"PreCool/"
else:
    path = prepath+"Cold/"
makedirs(path, exist_ok=True)

MsrNo=1 ## Take {MsrNo} measurements
MsrWait=1 ## wait {MsrWait} minutes

prog_bar=display('',display_id=True)

for i in range(MsrNo):
    start=datetime.now()
    Measurement_Msg=f"#  Measurement {i+1} - {Elsa.GetT('t4k')} - {start.strftime('%y%m%d %H%M')}\n"
    WriteLog(Measurement_Msg, path+'log.txt')
    
    for chn, device in enumerate(DeviceList):
        clear_output()
        prog_bar=display('',display_id=True)
        print(Measurement_Msg)
        if device:
            TestDevice(device, chn, path, params)
    WriteLog(f"Measurement {i+1} end - {Elsa.GetT('t4k')}. Duration: {int((datetime.now()-start).seconds/60)}:{(datetime.now()-start).seconds%60:02d}\n\n#######################\n", path + 'log.txt')

    start=datetime.now()
    clear_output()
    plt.close('all')
    prog_bar=display('',display_id=True)
    while (datetime.now()-start).seconds < MsrWait*60:
        prog_bar.update(f"Waiting: {int(MsrWait*60)-(datetime.now()-start).seconds} s")
        sleep(1)
    prog_bar.update("Measuring")
    
prog_bar.update("Pre measurement done")

'Waiting: 13 s'

#  Measurement 1 - 297.052 - 241129 1916

## Ch 1 TP1
Set IdxVgs
Vg=(0, -1.5, -0.02), Vd=-0.05, Ilim=0.001
Done IdxVgs. Duration: 39 s                                     
LIN=0.700, Vth=0.662 V, SS=20.56 mV/dec, miyf=344.5, miyf=378.9, theta1=2.301e-01, theta2=1.561e-01
297.031,0.700,0.662,20.56,344.5,378.9,2.301e-01,1.561e-01


KeyboardInterrupt: 

In [4]:
###### Pre measurement

if Elsa.GetT('t4k') > 150:
    path = prepath+"Vac/"
else:
    path = prepath+"Cold/"
makedirs(path, exist_ok=True)

MsrNo=1 ## Take {MsrNo} measurements
MsrWait=1 ## wait {MsrWait} minutes

prog_bar=display('',display_id=True)

for i in range(MsrNo):
    start=datetime.now()
    Measurement_Msg=f"#  Measurement {i+1} - {Elsa.GetT('t4k')} - {start.strftime('%y%m%d %H%M')}\n"
    WriteLog(Measurement_Msg, path+'log.txt')
    
    for chn, device in enumerate(DeviceList):
        clear_output()
        prog_bar=display('',display_id=True)
        print(Measurement_Msg)
        if device:
            TestDevice(device, chn, path, params, True)
    WriteLog(f"Measurement {i+1} end - {Elsa.GetT('t4k')}. Duration: {int((datetime.now()-start).seconds/60)}:{(datetime.now()-start).seconds%60:02d}\n\n#######################\n", path + 'log.txt')

    start=datetime.now()
    clear_output()
    plt.close('all')
    prog_bar=display('',display_id=True)
    while (datetime.now()-start).seconds < MsrWait*60:
        prog_bar.update(f"Waiting: {int(MsrWait*60)-(datetime.now()-start).seconds} s")
        sleep(1)
    prog_bar.update("Measuring")
    
prog_bar.update("Pre measurement done")

'Pre measurement done'

In [4]:
###### Warmup measurement

# if Elsa.GetT('t4k') > 150:
path = prepath+"Cooldown/"
# else:
#     path = prepath+"Warmup/"
makedirs(path, exist_ok=True)

freq_temp=10
MsrNo=3 ## Take {MsrNo} measurements
MsrWait=0.5 ## wait {MsrWait} minutes
failsafe_time=[60, 120, 120, 60] ## Maximum wait time between measurements (API crash failsafe)
failsafe_temp=[0, 120, 180, 300] ## is fitted from these temps and times
failsafe=np.polyfit(failsafe_time, failsafe_temp, 2)
clear_output()

prog_bar=display('',display_id=True)

current_temp=Elsa.GetT('t4k')

current_temp=Elsa.GetT('t4k')

if current_temp > 150: finish_temp=5
else: finish_temp=295

last_temp=150

while (last_temp != finish_temp):
    #loop until temp changes and is a multiple of freq_temp
    MaxWait=np.polyval(failsafe, last_temp) ## API crash failsafe calculation
    start=datetime.now()
    while (np.around(current_temp)==last_temp or np.around(current_temp)%freq_temp != 5):
        if (datetime.now()-start).seconds/60 > MaxWait: ## Failsafe trigger
            break
        for i in range(15):
            prog_bar.update(f"{datetime.now().strftime('%H:%M:%S')} - T = {format(current_temp, '.1f')} K")
            sleep(1)
    
        current_temp=Elsa.GetT('t4k')

    prog_bar.update("Measuring")

    last_temp=np.around(current_temp)
    
    for Msr in range(MsrNo):
        plt.close('all')
        clear_output()
        prog_bar=display('',display_id=True)

        start=datetime.now()
        WriteLog(f"# {current_temp} K - Measurement {Msr+1} - {datetime.now().strftime('%y%m%d %H%M')}\n", path+'log.txt')
        for chn, device in enumerate(DeviceList):
            if device:
                TestDevice(device, chn, path, params)
        WriteLog(f"{current_temp} K - Measurement {Msr+1} end . Duration: {int((datetime.now()-start).seconds/60)}:{(datetime.now()-start).seconds%60:02d}\n\n#######################\n", path + 'log.txt')
        
        INO.opench(1)

        pathp=path+'TP7'

        HP.SetDiode(0, 1.5, 0.05, Comp=0.5e-3)
        now=datetime.now().strftime('%y%m%d %H%M')
        Plot(HP.SingleSave(f"{pathp}/Diode - {now}.csv", timeout=30), 'Vf', ['Id', 'Is'])
        
        temp=format(Elsa.GetT('t4k'), '07.3f')
        rename(f"{pathp}/Diode - {now}.csv", f"{pathp}/Diode - {temp} - {now}.csv")
        rename(f"{pathp}/Diode - {now}.png", f"{pathp}/Diode - {temp} - {now}.png")

        current_temp=Elsa.GetT('t4k')
        start=datetime.now()
        while (datetime.now()-start).seconds < MsrWait*60:
            prog_bar.update(f"Waiting: {int(MsrWait*60)-(datetime.now()-start).seconds} s")
            sleep(1)
        prog_bar.update("Measuring")
prog_bar.update("Cooldown measurement done")

''

# 38.0682 K - Measurement 2 - 241201 0240



UnicodeDecodeError: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte

In [ ]:
finish_temp

295

In [ ]:
###### Post measurement

prog_bar=display('',display_id=True)

if Elsa.GetT('t4k') > 150:
    path = prepath+"PostCool/"
else:
    path = prepath+"Cold/"
makedirs(path, exist_ok=True)

MsrNo=3 ## Take {MsrNo} measurements
MsrWait=5 ## wait {MsrWait} minutes

for i in range(MsrNo):
    start=datetime.now()
    Measurement_Msg=f"#  Measurement {i+1} - {Elsa.GetT('t4k')} - {start.strftime('%y%m%d %H%M')}\n"
    WriteLog(Measurement_Msg, path+'log.txt')
    
    for chn, device in enumerate(DeviceList):
        clear_output()
        prog_bar=display('',display_id=True)
        print(Measurement_Msg)
        if device:
            TestDevice(device, chn, path, params, True)
    WriteLog(f"Measurement {i+1} end - {Elsa.GetT('t4k')}. Duration: {int((datetime.now()-start).seconds/60)}:{(datetime.now()-start).seconds%60:02d}\n\n#######################\n", path + 'log.txt')

    start=datetime.now()
    clear_output()
    plt.close('all')
    prog_bar=display('',display_id=True)
    while (datetime.now()-start).seconds < MsrWait*60:
        prog_bar.update(f"Waiting: {int(MsrWait*60)-(datetime.now()-start).seconds} s")
        sleep(1)    

print("Post measurement done")

'Waiting: 44 s'

#  Measurement 1 - 3.40316 - 241108 1419

## Ch 4 TN10
Set IdxVgs
Vg=(0, 1.5, 0.02), Vd=0.05, Ilim=0.001
Done IdxVgs. Duration: 32 s                                     
>>> Error: NaN values detected in your input data or the output of your objective/model function - fitting algorithms cannot handle this! Please read https://lmfit.github.io/lmfit-py/faq.html#i-get-errors-from-nan-in-my-fit-what-can-i-do for more information.
Vth=0.683 V
Set IdxVgs Sat
Vg=(0, 1.5, 0.02), Vd=1.5, Ilim=0.001
Done IdxVgs Sat. Duration: 32 s                                     
>>> Error: ufunc 'multiply' did not contain a loop with signature matching types (dtype('float64'), dtype('<U7')) -> None
n=0.000, Ispec=0.000e+00 A
Set IdxVds
Vd=(0, 1.5, 0.02), Vg=(0.6, 1.4, 0.2), Ilim=0.001
Done IdxVds. Duration: 215 s                                     s 15s | 15s 15s


In [ ]:
if Elsa.GetT('t4k') > 150:
        path = prepath+"PostCool/"
        makedirs(path, exist_ok=True)
        
        current_pres=Elsa.GetPCh(6)
        prog_bar=display('',display_id=True)
        while current_pres < 0.9:
                for i in range(15):
                        prog_bar.update(f"{datetime.now().strftime('%H:%M:%S')} - P = {format(current_pres, '.2f')} Torr")
                        sleep(1)
                current_pres=Elsa.GetPCh(6)
                if current_pres < 0.9:
                        sleep(5)
                        current_pres=Elsa.GetPCh(6)

        prog_bar=display('',display_id=True)

        MsrNo=3 ## Take {MsrNo} measurements
        MsrWait=5 ## wait {MsrWait} minutes

        for i in range(MsrNo):
                start=datetime.now()
                Measurement_Msg=f"#  Measurement {i+1} - {Elsa.GetT('t4k')} - {start.strftime('%y%m%d %H%M')}\n"
                WriteLog(Measurement_Msg, path+'log.txt')

                for chn, device in enumerate(DeviceList):
                        clear_output()
                        print(Measurement_Msg)
                        if device:
                                TestDevice(device, chn, path, params, True)
                WriteLog(f"Measurement {i+1} end - {Elsa.GetT('t4k')}. Duration: {int((datetime.now()-start).seconds/60)}:{(datetime.now()-start).seconds%60:02d}\n\n#######################\n", path + 'log.txt')

                start=datetime.now()
                clear_output()
                plt.close('all')
                prog_bar=display('',display_id=True)
                while (datetime.now()-start).seconds < MsrWait*60):
                        prog_bar.update(f"Waiting: {int(MsrWait*60)-(datetime.now()-start).seconds} s")
                        sleep(1)

        prog_bar.update("Measuring")

'Measuring'

In [ ]:
HP.close()
INO.close()